# SF Planning Permit Timelines
Who moves fastest through the SF Planning Department process, by project type?

## Data model
```
review_metrics  (d4jk-jw33)  b1_alt_id ──────────────► projects (qvu5-m3a2)  record_id
                                                                │
                                                         child_id (pipe-delimited)
                                                                │
                                                                ▼
                                              non_projects (y673-d69b)  record_id
                                              (ENV, CUA, VAR, PRL …)
```

In [ ]:
import pandas as pd
import requests
import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from io import StringIO

## Fetch datasets

In [ ]:
# Review metrics — all stages, all rows (≈70k)
url_metrics = (
    "https://data.sfgov.org/resource/d4jk-jw33.csv"
    "?$limit=100000"
    "&$order=b1_alt_id"
)
r = requests.get(url_metrics, timeout=120); r.raise_for_status()
df_metrics = pd.read_csv(StringIO(r.text))
print(f"review_metrics: {len(df_metrics):,} rows")
df_metrics.head(2)

In [ ]:
# Projects — all columns needed for applicant + project type clustering
url_projects = (
    "https://data.sfgov.org/resource/qvu5-m3a2.csv"
    "?$select=record_id,record_type,applicant,applicant_org,assigned_to_planner,"
    "record_status,project_decision,project_address,"
    "open_date,close_date,project_decision_date,"
    "new_construction,adu,change_of_use,demolition,additions,"
    "facade_alt,lot_line_adjust,leg_zone_change,other_prj_desc,"
    "environmental_review_type,number_of_units_net,number_of_units_prop,"
    "child_id"
    "&$limit=60000"
    "&$order=open_date%20DESC"
)
r = requests.get(url_projects, timeout=120); r.raise_for_status()
df_projects = pd.read_csv(StringIO(r.text))
print(f"projects: {len(df_projects):,} rows")
df_projects.head(2)

In [ ]:
# Non-Projects — child records (ENV, CUA, VAR, PRL, etc.) linked to projects via parent_id
url_nonprojects = (
    "https://data.sfgov.org/resource/y673-d69b.csv"
    "?$select=record_id,record_type,parent_id,applicant,applicant_org,"
    "record_status,open_date,close_date,project_address,description"
    "&$limit=100000"
    "&$order=open_date%20DESC"
)
r = requests.get(url_nonprojects, timeout=120); r.raise_for_status()
df_nonprojects = pd.read_csv(StringIO(r.text))
print(f"non_projects: {len(df_nonprojects):,} rows")
df_nonprojects["record_type"].value_counts().head(20)

## Load into DuckDB

In [ ]:
con = duckdb.connect()

con.execute("CREATE TABLE review_metrics  AS SELECT * FROM df_metrics")
con.execute("CREATE TABLE projects        AS SELECT * FROM df_projects")
con.execute("CREATE TABLE non_projects    AS SELECT * FROM df_nonprojects")

for t in ["review_metrics", "projects", "non_projects"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n:,} rows")

## Project type taxonomy
Since all projects share `record_type = PRJ`, we derive project type from the boolean flag columns.
Priority order: if multiple flags are set we assign the highest-complexity type.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW project_typed AS
SELECT
    record_id,
    applicant,
    applicant_org,
    assigned_to_planner,
    record_status,
    project_decision,
    project_address,
    TRY_CAST(open_date            AS DATE) AS open_date,
    TRY_CAST(close_date           AS DATE) AS close_date,
    TRY_CAST(project_decision_date AS DATE) AS decision_date,
    environmental_review_type,
    COALESCE(TRY_CAST(number_of_units_net AS INTEGER), 0)  AS units_net,
    COALESCE(TRY_CAST(number_of_units_prop AS INTEGER), 0) AS units_prop,
    -- Derive project type (priority: most complex first)
    CASE
        WHEN new_construction  = 'true' THEN 'New Construction'
        WHEN demolition        = 'true' THEN 'Demolition'
        WHEN leg_zone_change   = 'true' THEN 'Zoning Change'
        WHEN change_of_use     = 'true' THEN 'Change of Use'
        WHEN adu               = 'true' THEN 'ADU'
        WHEN additions         = 'true' THEN 'Addition'
        WHEN facade_alt        = 'true' THEN 'Facade Alteration'
        WHEN lot_line_adjust   = 'true' THEN 'Lot Line Adjustment'
        ELSE 'Other / Unclassified'
    END AS project_type,
    -- Size bucket for sub-clustering
    CASE
        WHEN COALESCE(TRY_CAST(number_of_units_net AS INTEGER), 0) = 0  THEN 'Non-residential / Unknown'
        WHEN COALESCE(TRY_CAST(number_of_units_net AS INTEGER), 0) <= 4  THEN '1–4 units'
        WHEN COALESCE(TRY_CAST(number_of_units_net AS INTEGER), 0) <= 20 THEN '5–20 units'
        ELSE '20+ units'
    END AS size_bucket
FROM projects
""")

con.execute("""
    SELECT project_type, size_bucket, COUNT(*) AS n
    FROM project_typed
    GROUP BY project_type, size_bucket
    ORDER BY n DESC
""").df()

## Master timeline view
Join projects → review metrics → non-project child record types.
Primary duration = `metric_value` (days) from the **accepted-to-approved** stage (planning dept's own SLA metric).
Secondary = calendar days from `open_date` to `close_date`.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW timeline AS
WITH
-- One row per project: pivot the two most useful metric stages
metrics_pivoted AS (
    SELECT
        b1_alt_id,
        MAX(CASE WHEN project_stage = 'accepted to approved'
                 THEN TRY_CAST(metric_value AS DOUBLE) END) AS days_accepted_to_approved,
        MAX(CASE WHEN project_stage = 'first plan review'
                 THEN TRY_CAST(metric_value AS DOUBLE) END) AS days_first_plan_review,
        MAX(CASE WHEN project_stage = 'accepted to approved'
                 THEN metric_outcome END)                    AS sla_outcome,
        MAX(CASE WHEN project_stage = 'accepted to approved'
                 THEN TRY_CAST(sla_value AS DOUBLE) END)    AS sla_target_days
    FROM review_metrics
    GROUP BY b1_alt_id
),
-- Child record types per project (pipe-delimited child_id → non_projects)
child_types AS (
    SELECT
        np.parent_id                                       AS project_id,
        STRING_AGG(DISTINCT np.record_type, ', '
                   ORDER BY np.record_type)                AS child_record_types,
        COUNT(DISTINCT np.record_id)                       AS child_count
    FROM non_projects np
    WHERE np.parent_id IS NOT NULL
    GROUP BY np.parent_id
)
SELECT
    pt.*,
    -- calendar days open → close
    DATE_DIFF('day', pt.open_date, COALESCE(pt.close_date, pt.decision_date)) AS calendar_days,
    -- planning dept SLA metrics
    mp.days_accepted_to_approved,
    mp.days_first_plan_review,
    mp.sla_outcome,
    mp.sla_target_days,
    -- child record context
    ct.child_record_types,
    ct.child_count
FROM project_typed pt
LEFT JOIN metrics_pivoted mp ON mp.b1_alt_id = pt.record_id
LEFT JOIN child_types     ct ON ct.project_id = pt.record_id
""")

con.execute("SELECT COUNT(*) AS total, COUNT(days_accepted_to_approved) AS has_a2a_metric, COUNT(calendar_days) AS has_calendar FROM timeline").df()

## Processing time distribution by project type

In [ ]:
dist = con.execute("""
    SELECT
        project_type,
        COUNT(*)                                            AS projects,
        COUNT(calendar_days)                               AS with_duration,
        ROUND(MEDIAN(calendar_days), 0)                    AS median_calendar_days,
        ROUND(AVG(calendar_days), 0)                       AS avg_calendar_days,
        ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP
              (ORDER BY calendar_days), 0)                 AS p25_days,
        ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP
              (ORDER BY calendar_days), 0)                 AS p75_days,
        ROUND(MEDIAN(days_accepted_to_approved), 0)        AS median_a2a_days,
        COUNT(CASE WHEN sla_outcome = 'Under Deadline'
                   THEN 1 END) * 100.0
            / NULLIF(COUNT(sla_outcome), 0)                AS pct_under_sla
    FROM timeline
    WHERE calendar_days BETWEEN 1 AND 3650   -- exclude outliers > 10 years
    GROUP BY project_type
    ORDER BY median_calendar_days
""").df()

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(dist["project_type"], dist["median_calendar_days"], color="#2563eb", label="Median")
ax.barh(dist["project_type"],
        dist["p75_days"] - dist["median_calendar_days"],
        left=dist["median_calendar_days"],
        color="#93c5fd", alpha=0.6, label="P25–P75 range")
ax.set_xlabel("Calendar Days (open → close)")
ax.set_title("Median Planning Timeline by Project Type", fontweight="bold")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

dist

## Fastest applicant orgs by project type
Minimum 5 closed projects per org×type cell to get a stable estimate.
Ranked by median calendar days within each project type.

In [ ]:
fastest_org = con.execute("""
WITH org_stats AS (
    SELECT
        project_type,
        COALESCE(NULLIF(TRIM(applicant_org), ''), '(individual / unknown)') AS org,
        COUNT(*)                                                AS projects,
        ROUND(MEDIAN(calendar_days), 0)                        AS median_days,
        ROUND(AVG(calendar_days), 0)                           AS avg_days,
        ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP
              (ORDER BY calendar_days), 0)                     AS p25_days,
        ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP
              (ORDER BY calendar_days), 0)                     AS p75_days,
        COUNT(CASE WHEN sla_outcome='Under Deadline' THEN 1 END)
            * 100.0 / NULLIF(COUNT(sla_outcome), 0)           AS pct_under_sla
    FROM timeline
    WHERE calendar_days BETWEEN 1 AND 3650
      AND record_status  = 'Closed'
    GROUP BY project_type, org
    HAVING COUNT(*) >= 5
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY project_type
                              ORDER BY median_days ASC) AS rank_in_type
    FROM org_stats
)
SELECT project_type, rank_in_type AS rank, org, projects,
       median_days, avg_days, p25_days, p75_days,
       ROUND(pct_under_sla, 1) AS pct_under_sla
FROM ranked
WHERE rank_in_type <= 5
ORDER BY project_type, rank_in_type
""").df()

fastest_org

In [ ]:
# Same analysis by individual applicant name (not org)
fastest_individual = con.execute("""
WITH ind_stats AS (
    SELECT
        project_type,
        COALESCE(NULLIF(TRIM(applicant), ''), '(unknown)') AS applicant,
        COUNT(*)                                            AS projects,
        ROUND(MEDIAN(calendar_days), 0)                    AS median_days,
        ROUND(AVG(calendar_days), 0)                       AS avg_days
    FROM timeline
    WHERE calendar_days BETWEEN 1 AND 3650
      AND record_status = 'Closed'
      AND NULLIF(TRIM(applicant), '') IS NOT NULL
    GROUP BY project_type, applicant
    HAVING COUNT(*) >= 3
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY project_type
                              ORDER BY median_days ASC) AS rank_in_type
    FROM ind_stats
)
SELECT project_type, rank_in_type AS rank, applicant, projects, median_days, avg_days
FROM ranked
WHERE rank_in_type <= 5
ORDER BY project_type, rank_in_type
""").df()

fastest_individual

In [ ]:
# Chart: fastest orgs per project type (top 5 project types by volume)
top_types = dist.nlargest(5, "projects")["project_type"].tolist()
subset = fastest_org[fastest_org["project_type"].isin(top_types)]

fig, axes = plt.subplots(1, len(top_types), figsize=(4 * len(top_types), 6), sharey=False)

for ax, ptype in zip(axes, top_types):
    data = subset[subset["project_type"] == ptype].sort_values("median_days")
    # truncate org names for display
    labels = [o[:28] + "…" if len(o) > 28 else o for o in data["org"]]
    ax.barh(labels, data["median_days"], color="#2563eb")
    # type median as reference line
    type_med = dist.loc[dist["project_type"] == ptype, "median_calendar_days"].values[0]
    ax.axvline(type_med, color="#dc2626", linestyle="--", linewidth=1.2,
               label=f"Type median ({type_med:.0f}d)")
    ax.set_title(ptype, fontweight="bold", fontsize=9)
    ax.set_xlabel("Median days")
    ax.legend(fontsize=7)
    ax.invert_yaxis()

plt.suptitle("Top 5 Fastest Orgs per Project Type (closed projects, min 5 each)\nRed line = overall type median",
             fontweight="bold")
plt.tight_layout()
plt.show()

## Child record type breakdown
Non-project children (ENV, CUA, VAR, etc.) indicate complexity — more/heavier children = longer reviews.
Useful for understanding why some project types take longer and for controlling comparisons.

In [ ]:
child_impact = con.execute("""
    SELECT
        project_type,
        child_record_types,
        COUNT(*)                        AS projects,
        ROUND(MEDIAN(calendar_days), 0) AS median_days,
        ROUND(AVG(calendar_days), 0)    AS avg_days
    FROM timeline
    WHERE calendar_days BETWEEN 1 AND 3650
      AND record_status = 'Closed'
    GROUP BY project_type, child_record_types
    HAVING COUNT(*) >= 3
    ORDER BY project_type, median_days
""").df()

# Duration by number of child records
child_count_impact = con.execute("""
    SELECT
        project_type,
        COALESCE(child_count, 0)         AS num_children,
        COUNT(*)                         AS projects,
        ROUND(MEDIAN(calendar_days), 0)  AS median_days
    FROM timeline
    WHERE calendar_days BETWEEN 1 AND 3650
      AND record_status = 'Closed'
    GROUP BY project_type, num_children
    ORDER BY project_type, num_children
""").df()

print("=== Median days by project type + child record types ===")
print(child_impact.head(40).to_string())
print("\n=== Duration grows with child record count ===")
print(child_count_impact.to_string())